# Notebook 14 — Continued Pretraining and Domain Adaptation

Continued pretraining, sometimes called domain-adaptive pretraining or CPT, is the stage where you keep training a base model on raw text from a target domain before asking it to follow instructions in that domain. The goal is not to teach a single behavior. The goal is to shift the model's language prior toward the terminology, style, and factual regularities of a corpus.

## What you will build

- a decision framework for choosing CPT versus supervised fine-tuning
- a small raw-text corpus for a network operations domain
- chunking and token-budget utilities for local Colab experiments
- a tiny continued pretraining run using raw text instead of instruction pairs
- before versus after retention checks on domain and general text
- a practical note on embeddings and lm_head learning rates

## Why this notebook matters

Supervised fine-tuning is the default post-training move when you want specific behaviors or output formats. CPT becomes attractive when the model is repeatedly confused by domain vocabulary, document style, abbreviations, or discourse patterns that are hard to inject through prompt engineering alone. In practice, you should treat CPT as a domain transfer lever, not as a generic first response to every quality problem.

In [ ]:
# --- Google Colab Pro Fine-Tuning + Evaluation Setup ---
%%capture
!pip install -q --upgrade pip
!pip install -q unsloth datasets trl peft accelerate bitsandbytes pandas matplotlib scikit-learn

import json
import math
import random
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from datasets import Dataset, DatasetDict, load_dataset
from google.colab import drive
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

drive.mount('/content/drive')

CACHE_DIR = "/content/drive/MyDrive/models"
OUTPUT_DIR = "/content/drive/MyDrive/castalia-finetuning"
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    cache_dir=CACHE_DIR,
)

tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

def formatting_prompts_func(batch):
    return {"text": batch["text"]}

print("✓ Fine-tuning stack ready")
print("  Model:", MODEL_NAME)
print("  CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("  BF16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

In [ ]:
from collections import Counter, defaultdict
from statistics import mean

random.seed(14)

ARTIFACT_DIR = Path("artifacts") / "notebook_14_continued_pretraining"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def to_markdown_table(rows, columns):
    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(str(row.get(column, "")) for column in columns) + " |")
    return "\n".join([header, divider, *body])

def approx_token_count(text):
    return max(1, int(len(text.split()) * 1.25))

def ascii_bar(value, width=16):
    value = max(0.0, min(1.0, float(value)))
    filled = int(round(value * width))
    return "█" * filled + "·" * (width - filled)

print("Artifact directory:", ARTIFACT_DIR.resolve())

## Step 1: Compare adaptation levers

Use the lightest tool that solves the problem. CPT is powerful, but it is slower, riskier, and easier to misuse than prompting, retrieval, or supervised fine-tuning.

In [ ]:
adaptation_options = pd.DataFrame(
    [
        {
            "strategy": "Prompting",
            "best_for": "one-off control and rapid iteration",
            "data_shape": "no new training data",
            "compute_cost": "lowest",
            "risk": "low",
        },
        {
            "strategy": "RAG",
            "best_for": "fresh or private knowledge grounding",
            "data_shape": "indexed documents",
            "compute_cost": "low to medium",
            "risk": "low to medium",
        },
        {
            "strategy": "SFT",
            "best_for": "task behavior, formatting, refusal style, tool use",
            "data_shape": "instruction and response pairs",
            "compute_cost": "medium",
            "risk": "medium",
        },
        {
            "strategy": "CPT",
            "best_for": "domain vocabulary, discourse patterns, style transfer",
            "data_shape": "large raw text corpora",
            "compute_cost": "medium to high",
            "risk": "high if retention is ignored",
        },
    ]
)

display(adaptation_options)

## Step 2: Score when CPT is justified

In practice, CPT is usually justified only when several signals line up at once: a lot of raw text, repeated domain vocabulary failures, weak tokenizer or embedding coverage for target terms, and evidence that retrieval alone is not enough.

In [ ]:
def cpt_justification_score(row):
    score = 0.0
    score += 0.30 * row["raw_text_abundance"]
    score += 0.20 * row["term_shift"]
    score += 0.15 * row["style_shift"]
    score += 0.20 * row["retrieval_gap"]
    score += 0.15 * row["long_horizon_value"]
    return round(score, 3)

scenarios = pd.DataFrame(
    [
        {
            "scenario": "Support bot needs a new JSON format",
            "raw_text_abundance": 0.1,
            "term_shift": 0.1,
            "style_shift": 0.2,
            "retrieval_gap": 0.1,
            "long_horizon_value": 0.2,
        },
        {
            "scenario": "Model struggles with telecom runbooks and change tickets",
            "raw_text_abundance": 0.9,
            "term_shift": 0.9,
            "style_shift": 0.8,
            "retrieval_gap": 0.6,
            "long_horizon_value": 0.8,
        },
        {
            "scenario": "Need short medical coding answers from labeled examples",
            "raw_text_abundance": 0.3,
            "term_shift": 0.5,
            "style_shift": 0.3,
            "retrieval_gap": 0.2,
            "long_horizon_value": 0.4,
        },
        {
            "scenario": "Frequent legal abbreviation errors across thousands of filings",
            "raw_text_abundance": 0.95,
            "term_shift": 0.85,
            "style_shift": 0.7,
            "retrieval_gap": 0.55,
            "long_horizon_value": 0.9,
        },
    ]
)

scenarios["cpt_score"] = scenarios.apply(cpt_justification_score, axis=1)
scenarios["recommended_move"] = scenarios["cpt_score"].apply(
    lambda value: "CPT candidate" if value >= 0.65 else "Prefer SFT or RAG"
)

display(scenarios[["scenario", "cpt_score", "recommended_move"]])

## Step 3: Build a raw-text domain corpus

For CPT, the basic unit is raw text, not prompt and answer pairs. We will use short network-operations notes and runbook fragments so the notebook stays local and Colab-realistic while still showing the core workflow.

In [ ]:
domain_documents = [
    {
        "source": "runbook",
        "text": "When packet loss rises after a deploy, compare the canary shard against the stable shard before paging the transport team. If only the canary is affected, roll back the release and keep the backbone unchanged.",
    },
    {
        "source": "postmortem",
        "text": "The sev2 incident started with a BGP flap on the edge routers. Alert fatigue delayed acknowledgement, so the escalation policy now requires paging the on-call within three minutes when flap dampening exceeds the threshold.",
    },
    {
        "source": "change-ticket",
        "text": "Every maintenance window needs a customer communication draft, rollback owner, and a health-check command. Do not close the change until latency, error rate, and queue depth return to the pre-change baseline.",
    },
    {
        "source": "runbook",
        "text": "For cache saturation, drain one node at a time, confirm hit ratio recovery, and watch write amplification. If eviction storms continue after the drain, fail over the read-heavy tenants before scaling the cluster.",
    },
    {
        "source": "ops-guide",
        "text": "Traffic shadowing is preferred before global rollout. Mirror production reads to the candidate service, compare p95 latency and malformed response counts, and stop the shadow if tenant isolation metrics degrade.",
    },
    {
        "source": "incident-review",
        "text": "A noisy alert should not page the primary unless the symptom persists across two windows. The better fix is to tighten the detector and attach a runbook note that explains probable causes and triage commands.",
    },
    {
        "source": "runbook",
        "text": "If the queue backlog grows while CPU remains low, inspect upstream retry storms and stuck consumers before adding replicas. Blind autoscaling can hide poison messages and delay the real fix.",
    },
    {
        "source": "ops-guide",
        "text": "Regional failover exercises must document DNS TTL, warm standby lag, and cache invalidation timing. Success is not just traffic movement. Success means customers avoid stale data and auth loops during recovery.",
    },
    {
        "source": "postmortem",
        "text": "The root cause was an incomplete schema rollout. One service wrote the new enum values while another still rejected them, causing retries, duplicate jobs, and a backlog on the reconciliation queue.",
    },
    {
        "source": "change-ticket",
        "text": "Database migrations on Friday require director approval. If a migration touches auth tables, stage the migration during low volume, precompute rollback queries, and monitor replication lag every five minutes.",
    },
    {
        "source": "incident-review",
        "text": "Customer impact is measured by failed requests and degraded workflows, not only by host-level metrics. The summary should name the affected product path, the blast radius, and the workaround if one exists.",
    },
    {
        "source": "runbook",
        "text": "Before draining a pod, verify that downstream leases have enough headroom. Lease starvation can look like latency regressions even when the application is healthy.",
    },
]

domain_probes = [
    "Explain what a canary shard rollback protects against in a network rollout.",
    "Why can blind autoscaling be dangerous during a retry storm?",
    "Summarize why replication lag matters during auth-table migrations.",
    "What should an incident summary include besides host-level metrics?",
]

general_probes = [
    "Write a short explanation of why rainfall patterns matter to farmers.",
    "Explain the difference between correlation and causation in plain language.",
    "Give three tips for learning a new language consistently.",
    "Summarize how photosynthesis works for a middle-school student.",
]

print("Domain documents:", len(domain_documents))
print("Domain probes:", len(domain_probes))
print("General probes:", len(general_probes))

## Step 4: Inspect corpus scale and token budget

Even in a tiny teaching notebook, it helps to quantify how much text you actually have. Token totals drive how many effective steps you can afford and whether CPT is worth running at all.

In [ ]:
corpus_stats = pd.DataFrame(
    [
        {
            "source": item["source"],
            "approx_tokens": approx_token_count(item["text"]),
            "characters": len(item["text"]),
        }
        for item in domain_documents
    ]
)

summary = {
    "documents": len(corpus_stats),
    "total_approx_tokens": int(corpus_stats["approx_tokens"].sum()),
    "mean_tokens_per_doc": round(corpus_stats["approx_tokens"].mean(), 1),
    "max_tokens_per_doc": int(corpus_stats["approx_tokens"].max()),
}

display(corpus_stats)
print(summary)

scaling_plan = pd.DataFrame(
    [
        {"multiplier": "1x", "effective_tokens": summary["total_approx_tokens"]},
        {"multiplier": "5x", "effective_tokens": summary["total_approx_tokens"] * 5},
        {"multiplier": "20x", "effective_tokens": summary["total_approx_tokens"] * 20},
    ]
)
display(scaling_plan)

corpus_stats.to_csv(ARTIFACT_DIR / "corpus_stats.csv", index=False)

## Step 5: Chunk the raw text for continued pretraining

For CPT we keep the text as text. There is no assistant-only masking here. We simply prepare contiguous chunks that the model can predict token by token.

In [ ]:
def chunk_words(text, target_words=70, stride_words=55):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        chunk = words[start : start + target_words]
        if len(chunk) < 18:
            break
        chunks.append(" ".join(chunk))
        if start + target_words >= len(words):
            break
        start += stride_words
    return chunks

cpt_rows = []
for item in domain_documents:
    for chunk_index, chunk in enumerate(chunk_words(item["text"])):
        cpt_rows.append(
            {
                "source": item["source"],
                "chunk_id": f'{item["source"]}_{chunk_index}',
                "text": chunk,
                "approx_tokens": approx_token_count(chunk),
            }
        )

cpt_dataset = Dataset.from_list(cpt_rows)
cpt_split = cpt_dataset.train_test_split(test_size=0.2, seed=14)

print("CPT rows:", len(cpt_rows))
display(pd.DataFrame(cpt_rows).head(8))
print("Train rows:", len(cpt_split["train"]))
print("Eval rows:", len(cpt_split["test"]))

## Step 6: Inspect vocabulary concentration

One of the strongest arguments for CPT is repeated exposure to jargon that the base model only half knows. We can audit that pressure by counting domain-heavy terms.

In [ ]:
domain_terms = [
    "canary",
    "shard",
    "backlog",
    "replication",
    "lag",
    "dampening",
    "tenant",
    "rollback",
    "failover",
    "queue",
    "eviction",
]

term_counts = Counter()
for item in domain_documents:
    lowered = item["text"].lower()
    for term in domain_terms:
        term_counts[term] += lowered.count(term)

term_df = pd.DataFrame(
    [{"term": term, "count": term_counts[term]} for term in domain_terms]
).sort_values("count", ascending=False)

display(term_df)
print("Sample chunk preview:")
for row in cpt_rows[:3]:
    print("-", row["text"])

## Step 7: Embedding and lm_head note

If you unfreeze embeddings or lm_head during CPT, use a smaller learning rate than the main transformer body. Those layers strongly affect lexical behavior and can drift quickly. On small hardware, many teams stay adapter-first or selectively unfreeze only a narrow set of weights.

In [ ]:
interesting_params = []
for name, param in model.named_parameters():
    if any(key in name for key in ["embed_tokens", "lm_head", "lora_A", "lora_B"]):
        interesting_params.append(
            {
                "name": name,
                "trainable": bool(param.requires_grad),
                "numel": int(param.numel()),
            }
        )

interesting_df = pd.DataFrame(interesting_params)
display(interesting_df.head(20))

optimizer_plan = pd.DataFrame(
    [
        {
            "group": "embeddings",
            "pattern": "embed_tokens",
            "suggested_lr": 5e-6,
            "reason": "protect lexical stability while adapting domain terms",
        },
        {
            "group": "lm_head",
            "pattern": "lm_head",
            "suggested_lr": 5e-6,
            "reason": "avoid over-steering the output distribution",
        },
        {
            "group": "adapter layers",
            "pattern": "lora_*",
            "suggested_lr": 1e-4,
            "reason": "main adaptation capacity for a Colab-scale run",
        },
    ]
)

display(optimizer_plan)

## Step 8: Define retention checks before training

We want two signals:

1. domain loss should go down after CPT
2. general loss should not degrade too far

The easiest notebook-friendly proxy is average token loss on small held-out texts.

In [ ]:
domain_holdout_texts = [
    "During failover drills, track DNS TTL and cache invalidation timing so customers do not see stale auth sessions.",
    "A change ticket is not complete until latency, error rate, and backlog return to baseline after rollout.",
    "Retry storms can hide poison messages, so queue growth with low CPU should trigger consumer inspection before scaling.",
]

general_holdout_texts = [
    "A coral reef supports many species because it offers food, shelter, and breeding space.",
    "Regular practice helps language learners because repetition strengthens recall and confidence.",
    "Correlation does not prove causation because two events can move together for other reasons.",
]

def average_loss(texts):
    model.eval()
    losses = []
    device = next(model.parameters()).device
    for text in texts:
        tokens = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        tokens = {key: value.to(device) for key, value in tokens.items()}
        with torch.no_grad():
            outputs = model(**tokens, labels=tokens["input_ids"])
        losses.append(float(outputs.loss.detach().cpu()))
    return round(mean(losses), 4)

baseline_metrics = {
    "domain_loss_before": average_loss(domain_holdout_texts),
    "general_loss_before": average_loss(general_holdout_texts),
}

baseline_metrics

## Step 9: Capture baseline generations on domain prompts

Loss is useful, but it is also helpful to read a few outputs. We keep prompts small to stay realistic for Colab.

In [ ]:
generation_prompts = [
    "Explain why queue backlog with low CPU can point to retry storms.",
    "Summarize what a strong maintenance window checklist should include.",
    "Why is traffic shadowing safer than a full rollout?",
]

def generate_completion(prompt, max_new_tokens=90):
    FastLanguageModel.for_inference(model)
    device = next(model.parameters()).device
    messages = [
        {"role": "system", "content": "You are a careful network operations mentor."},
        {"role": "user", "content": prompt},
    ]
    rendered = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(rendered, return_tensors="pt", truncation=True, max_length=512)
    inputs = {key: value.to(device) for key, value in inputs.items()}
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
    )
    answer_tokens = outputs[0][inputs["input_ids"].shape[1] :]
    return tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()

baseline_generations = [
    {"prompt": prompt, "response_before": generate_completion(prompt)}
    for prompt in generation_prompts
]

display(pd.DataFrame(baseline_generations))

## Step 10: Configure a tiny continued pretraining run

This is intentionally small. The point is to show the mechanics of CPT on local hardware, not to claim that a dozen chunks replaces a real corpus.

In [ ]:
training_args = TrainingArguments(
    output_dir=str(ARTIFACT_DIR / "cpt_adapter"),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    warmup_steps=4,
    max_steps=24,
    logging_steps=4,
    evaluation_strategy="steps",
    eval_steps=6,
    save_strategy="steps",
    save_steps=12,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=14,
    report_to="none",
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=cpt_split["train"],
    eval_dataset=cpt_split["test"],
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,
    args=training_args,
)

print("Trainer ready with", len(cpt_split["train"]), "training rows")

## Step 11: Run the local CPT slice

If this still feels heavy for your session, reduce max_steps or shrink the chunk set. In a real project you would also checkpoint losses and save adapter metadata after each run.

In [ ]:
train_result = trainer.train()
trainer.save_model()

training_log = pd.DataFrame(trainer.state.log_history)
training_log.to_csv(ARTIFACT_DIR / "training_log.csv", index=False)

display(training_log.tail(8))

## Step 12: Measure after-training loss

The target pattern is domain loss down, general loss roughly stable. If general loss rises sharply, you are probably overtraining or using a bad mixture.

In [ ]:
after_metrics = {
    "domain_loss_after": average_loss(domain_holdout_texts),
    "general_loss_after": average_loss(general_holdout_texts),
}

comparison = pd.DataFrame(
    [
        {
            "split": "domain",
            "before": baseline_metrics["domain_loss_before"],
            "after": after_metrics["domain_loss_after"],
        },
        {
            "split": "general",
            "before": baseline_metrics["general_loss_before"],
            "after": after_metrics["general_loss_after"],
        },
    ]
)
comparison["delta"] = comparison["after"] - comparison["before"]

display(comparison)

ax = comparison.set_index("split")[["before", "after"]].plot(
    kind="bar",
    figsize=(8, 4),
    title="Held-out loss before vs after CPT",
)
ax.set_ylabel("Average loss")
plt.show()

## Step 13: Inspect prompt completions after CPT

Read the outputs, not just the metrics. Stronger CPT often shows up as crisper use of terms like failover, canary, rollback, backlog, and tenant isolation.

In [ ]:
post_generations = [
    {"prompt": prompt, "response_after": generate_completion(prompt)}
    for prompt in generation_prompts
]

generation_review = pd.DataFrame(baseline_generations).merge(
    pd.DataFrame(post_generations),
    on="prompt",
)

def term_hit_count(text):
    lowered = text.lower()
    return sum(1 for term in domain_terms if term in lowered)

generation_review["term_hits_before"] = generation_review["response_before"].apply(term_hit_count)
generation_review["term_hits_after"] = generation_review["response_after"].apply(term_hit_count)

display(generation_review)
generation_review.to_csv(ARTIFACT_DIR / "generation_review.csv", index=False)

## Step 14: Compare CPT versus SFT use cases

The easiest mistake is using CPT to solve a problem that is really about behavior specification. Use CPT when the model needs to absorb a domain. Use SFT when the model needs to perform a task or follow a policy.

In [ ]:
def route_training_strategy(needs):
    if needs["format_control"] >= 0.7 or needs["tool_behavior"] >= 0.7:
        return "Prefer SFT"
    if needs["raw_text_corpus"] >= 0.7 and needs["term_shift"] >= 0.7:
        return "Prefer CPT"
    if needs["knowledge_freshness"] >= 0.7:
        return "Prefer RAG"
    return "Start with prompting and evals"

decision_cases = pd.DataFrame(
    [
        {
            "case": "Need XML incident output",
            "format_control": 0.9,
            "tool_behavior": 0.5,
            "raw_text_corpus": 0.2,
            "term_shift": 0.2,
            "knowledge_freshness": 0.3,
        },
        {
            "case": "Legal abbreviations and filing style feel foreign to the model",
            "format_control": 0.2,
            "tool_behavior": 0.1,
            "raw_text_corpus": 0.95,
            "term_shift": 0.9,
            "knowledge_freshness": 0.4,
        },
        {
            "case": "Need answers grounded in a changing policy wiki",
            "format_control": 0.2,
            "tool_behavior": 0.3,
            "raw_text_corpus": 0.5,
            "term_shift": 0.4,
            "knowledge_freshness": 0.95,
        },
        {
            "case": "Need polite refusal style and escalation language",
            "format_control": 0.8,
            "tool_behavior": 0.8,
            "raw_text_corpus": 0.2,
            "term_shift": 0.2,
            "knowledge_freshness": 0.3,
        },
    ]
)

decision_cases["recommended_move"] = decision_cases.apply(route_training_strategy, axis=1)
display(decision_cases[["case", "recommended_move"]])

## Key takeaways

- CPT is justified when the model needs broad domain transfer, not just a new response format.
- Raw text corpora are the main asset. Instruction pairs are optional at this stage.
- Embeddings and lm_head can be high-impact and high-risk, so smaller learning rates are common.
- Always pair CPT with retention checks on general text and later task-level SFT where needed.
- In most practical pipelines, CPT shifts the language prior first, then SFT teaches the exact behaviors you care about.